# Multiple Linear Regression in Beaufort Sea

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import statsmodels.api as sm

In [35]:
# define all vars
SSS = salt_august_avg.isel(k=0)
S_20m = salt_august_avg.isel(k=10)
SST = theta_august_avg.isel(k=0)
T_20m = theta_august_avg.isel(k=10)
strat = delta_sigma_20m

In [36]:
# look at shape
print(SSS.shape)
print(S_20m.shape)
print(SST.shape)
print(T_20m.shape)
print(strat.shape)

(7, 1080, 1800)
(7, 1080, 1800)
(7, 1080, 1800)
(7, 1080, 1800)
(7, 1080, 1800)


In [37]:
# see how correlated these terms are to one another

In [47]:
# First flatten 3D variables into 1D arrays
SSS_flat = SSS.values.flatten()
S_20m_flat = S_20m.values.flatten()
SST_flat = SST.values.flatten()
T_20m_flat = T_20m.values.flatten()
strat_flat = strat.values.flatten()

In [45]:
# Create a DataFrame for pairwise correlation
df = pd.DataFrame({
    "SSS": SSS_flat,
    "S_20m": S_20m_flat,
    "SST": SST_flat,
    "T_20m": T_20m_flat
})

In [46]:
# Drop rows with NaN values (e.g., land or missing data)
df = df.dropna()

# Compute pairwise correlations
correlation_matrix = df.corr()

# Display the correlation matrix
print("Pairwise Correlation Matrix:")
print(correlation_matrix)

Pairwise Correlation Matrix:
            SSS     S_20m       SST     T_20m
SSS    1.000000  0.452338 -0.153988 -0.099554
S_20m  0.452338  1.000000  0.049772  0.047262
SST   -0.153988  0.049772  1.000000  0.804500
T_20m -0.099554  0.047262  0.804500  1.000000


#### Run OLS analysis

In [74]:
# Create a data matrix (independent variables)
X = np.column_stack([SSS_flat, S_20m_flat, SST_flat])
X = sm.add_constant(X)  # Add intercept

# Fit regression model
model = sm.OLS(strat_flat, X, missing='drop')  # Handles NaN values
results = model.fit()

In [76]:
# Get coefficients
coefficients = results.params

print("Coefficients (a, b, c):", coefficients[1:])

Coefficients (a, b, c): [ 0.80086618 -0.80067484 -0.02746309]


In [77]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                 1.775e+09
Date:                Mon, 02 Dec 2024   Prob (F-statistic):               0.00
Time:                        20:58:37   Log-Likelihood:             5.1778e+06
No. Observations:             3609641   AIC:                        -1.036e+07
Df Residuals:                 3609637   BIC:                        -1.036e+07
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0634      0.001    -88.820      0.000      -0.065      -0.062
x1             0.8009   1.13e-05   7.11e+04      0.000       0.801       0.801
x2            -0.8007   2.54e-05  -3.15e+04      0.000      -0.801      -0.801
x3            -0.0275   1.51e-05  -1815.973      0.000      -0.027      -0.027
==============================================================================
Omnibus:                  2429712.081   Durbin-Watson:                   0.051
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         99015069.966
Skew:                           2.715   Prob(JB):                         0.00
Kurtosis:                      28.077   Cond. No.                     1.01e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.01e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""